# 📊 Crop Yield Prediction Regressor Development
### Agricultural Productivity Estimation with Random Forest & XGBoost

This notebook details the continuous regression modeling of crop yields (kg/ha):
- **Exploratory Data Analysis on 198,000+ Records**
- **State Extraction & Anomaly Handling**
- **Categorical One-Hot Encoding & Feature Scaling Pipeline**
- **Model Training: Random Forest vs XGBoost Regressors**
- **Performance Metrics (R², RMSE, MAE)**
- **Actual vs Predicted Diagnostic Plots**
- **Model Serialization**

In [1]:
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

data_path = (
    "../datasets/crop_yield_dataset.csv"
    if os.path.exists("../datasets/crop_yield_dataset.csv")
    else "datasets/crop_yield_dataset.csv"
)
df = pd.read_csv(data_path)
print(f"Crop Yield Dataset Shape: {df.shape}")
df.head()

In [2]:
# Clean and Extract State from Region
def extract_state(val):
    if not isinstance(val, str):
        return "All India"
    state = val.split(",")[-1].strip()
    m = {"AndhraPradesh": "Andhra Pradesh", "Himachal  Pradesh": "Himachal Pradesh"}
    return m.get(state, state)


df["State"] = df["Region"].apply(extract_state)
df = df[df["Yield"] > 0].copy()
df["Temperature"] = df["Temperature"].clip(lower=5.0, upper=50.0)
df["pH"] = df["pH"].clip(lower=3.0, upper=10.0)

print(f"Unique States ({df['State'].nunique()}):", df["State"].unique()[:10])
print(f"Unique Crops ({df['Crop'].nunique()}):", sorted(df["Crop"].unique()))
df.describe()

In [3]:
# Yield Distribution across Major Crops
plt.figure(figsize=(14, 6))
major_crops = [
    "Rice",
    "Wheat",
    "Maize",
    "Chickpea",
    "Cotton",
    "Sugarcane",
    "Pigeonpea",
    "Groundnut",
]
sns.boxplot(x="Crop", y="Yield", data=df[df["Crop"].isin(major_crops)], palette="Set2")
plt.yscale("log")
plt.title("Yield Distribution across Major Indian Crops (Log Scale)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [4]:
# Preprocessor Construction & Train-Test Split
num_features = [
    "Temperature",
    "Rainfall",
    "Humidity",
    "pH",
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "Fertilizer",
    "Pesticide",
]
cat_features = ["Crop", "Season", "Soil Type", "State"]

# Representative sample for notebook speed
sample_df = df.sample(n=40000, random_state=42).reset_index(drop=True)
X_df = sample_df[num_features + cat_features]
y = sample_df["Yield"].values

X_train_df, X_test_df, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ]
)

X_train = preprocessor.fit_transform(X_train_df)
X_test = preprocessor.transform(X_test_df)
print(f"Transformed Features Shape: {X_train.shape}")

In [5]:
# 1. Random Forest Regressor
rf = RandomForestRegressor(n_estimators=60, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
print(f"Random Forest R2: {rf_r2:.4f}, RMSE: {rf_rmse:.2f} kg/ha")

In [6]:
# 2. XGBoost Regressor
xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.08,
    max_depth=7,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_r2 = r2_score(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
print(f"XGBoost R2: {xgb_r2:.4f}, RMSE: {xgb_rmse:.2f} kg/ha")

In [7]:
# Actual vs Predicted Scatter Diagnostics
plt.figure(figsize=(7, 7))
plt.scatter(y_test[:400], xgb_preds[:400], alpha=0.5, color="#2E7D32", edgecolors="k", s=35)
min_val = min(min(y_test[:400]), min(xgb_preds[:400]))
max_val = max(max(y_test[:400]), max(xgb_preds[:400]))
plt.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label="Ideal 1:1 Identity")
plt.title("Yield Prediction — Actual vs Predicted Yield (kg/ha)")
plt.xlabel("Ground-Truth Yield (kg/ha)")
plt.ylabel("Model Predicted Yield (kg/ha)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [8]:
# Model Serialization
models_dir = "../models" if os.path.exists("../models") else "models"
joblib.dump(xgb, os.path.join(models_dir, "yield_prediction_model.pkl"))
joblib.dump(
    {"preprocessor": preprocessor, "features": num_features + cat_features},
    os.path.join(models_dir, "yield_prediction_preprocessor.pkl"),
)
print("Saved Yield Prediction Regressor successfully!")